# 🚀 KernelGemma: eBPF Unsloth Fine-Tuning Pipeline
This notebook fine-tunes Gemma 4 E4B to write verifier-safe Linux eBPF code. Ensure Runtime is set to **T4 GPU** and kernelgemma_dataset.jsonl is uploaded.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

# Load Gemma 4 E4B
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configure QLoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ Model and LoRA Adapters Loaded!")

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Enforce Gemma Chat ML Formatting
tokenizer = get_chat_template(tokenizer, chat_template="gemma")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

# Load the generated JSONL dataset
dataset = load_dataset("json", data_files="kernelgemma_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Dataset Loaded: {len(dataset)} rows perfectly formatted.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 125, # Optimized for 500 rows on T4 GPU
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🔥 KERNELGEMMA TRAINING STARTED 🔥")
trainer_stats = trainer.train()
print("✅ TRAINING COMPLETE!")

In [ ]:
# Save the model locally in the Colab instance
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

print("🚀 COMPILING TO GGUF (Q4_K_M) FOR EDGE INFERENCE 🚀")
# Export to 4-bit GGUF for llama.cpp execution on standard CPUs/GPUs
model.save_pretrained_gguf("kernelgemma-e4b", tokenizer, quantization_method = "q4_k_m")
print("✅ GGUF EXPORT COMPLETE! Download the .gguf file from the left sidebar.")